In [ ]:
# 1. Importar bibliotecas
import csv
import os
from pathlib import Path
import tempfile
import unittest

print('Ambiente Python OK')

# 2. Configurar ambiente e dados de teste
BASE_DIR = Path.cwd()
TEST_DIR = BASE_DIR / 'tmp_validacao_split_csv'
TEST_DIR.mkdir(exist_ok=True)

# Criar um arquivo de exemplo com cabeçalho e alguns dados
input_path = TEST_DIR / 'entrada.csv'
input_path.write_text('codigo|nome\n1|Ana\n2|Bruno\n3|Carla\n4|Diana\n', encoding='utf-8')

print(f'Pasta de trabalho: {BASE_DIR}')
print(f'Arquivo criado: {input_path}')

# 3. Implementar função principal
import importlib.util
module_path = BASE_DIR / 'split_csv_em_lotes.py'
spec = importlib.util.spec_from_file_location('split_csv_em_lotes', module_path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
split_csv_to_batches = mod.split_csv_to_batches

# 4. Criar testes unitários
class TestSplitCsv(unittest.TestCase):
    def test_divisao_em_lotes_preserva_cabecalho(self):
        output_dir = TEST_DIR / 'partes'
        generated = split_csv_to_batches(str(input_path), str(output_dir), rows_per_file=2, delimiter='|')
        self.assertEqual(len(generated), 2)

        with open(generated[0], 'r', encoding='utf-8', newline='') as f:
            first = list(csv.reader(f, delimiter='|'))
        with open(generated[1], 'r', encoding='utf-8', newline='') as f:
            second = list(csv.reader(f, delimiter='|'))

        self.assertEqual(first, [['codigo', 'nome'], ['1', 'Ana'], ['2', 'Bruno']])
        self.assertEqual(second, [['codigo', 'nome'], ['3', 'Carla'], ['4', 'Diana']])

    def test_arquivo_vazio_lanca_erro(self):
        empty_path = TEST_DIR / 'vazio.csv'
        empty_path.write_text('', encoding='utf-8')
        with self.assertRaises(ValueError):
            split_csv_to_batches(str(empty_path), str(TEST_DIR / 'vazio_partes'), delimiter='|')

    def test_arquivo_inexistente_lanca_erro(self):
        with self.assertRaises(FileNotFoundError):
            split_csv_to_batches(str(TEST_DIR / 'nao_existe.csv'), str(TEST_DIR / 'x'), delimiter='|')

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestSplitCsv)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'Há falhas nos testes de validação.'

print('Todos os testes passaram.')
